In [1]:
import os
print(os.listdir())


['.anaconda', '.android', '.cache', '.conda', '.condarc', '.continuum', '.ipynb_checkpoints', '.ipython', '.jupyter', '.streamlit', '.vscode', '.wdm', '04_KMM_BOQ', '90342907', 'aai_boq', 'AFD.ipynb', 'aiib.ipynb', 'aiib_f...ipynb', 'alldata_fetched.ipynb', 'all_boq_dipipe_onesheet.ipynb', 'all_boq_to_onesheet_find CLASS.ipynb', 'AMRUT-31.ipynb', 'Amrut_teender247.ipynb', 'amrut_tender_scriping.ipynb', 'anaconda3', 'anaconda_projects', 'Andaman And Nicobar Islands_All_BOQ_Live', 'Andhra Pradesh_All_BOQ_Live', 'Andhra_Pradesh_All_BOQ_Live', 'AppData', 'Application Data', 'Arunachal Pradesh_all_BOQ_live', 'Arunachal_Oneyeardata_with_BOQ', 'Arunachal_Pradesh_all_BOQ_live', 'assam.ipynb', 'Assam_govt_past_departmentwise.ipynb', 'Assam_Pastdata_Boq_Data_Extracted', 'Bengal.ipynb', 'Bihar_Eprocurment.ipynb', 'BOQ', 'BOQFilter .ipynb', 'BOQSHEET TO CLASS FIND.ipynb', 'BOQ_527123', 'BOQ_Downloads', 'BOQ_Downloads mp', 'BOQ_download_odisha_dept_site.ipynb', 'BOQ_download_odisha_govt_site.ipynb'

In [4]:
from selenium import webdriver
import json
import time
from selenium.common.exceptions import WebDriverException

driver = webdriver.Chrome()
driver.get("https://nexizo.ai")   # open the site domain first
time.sleep(2)

# Load cookies.json
with open("cookies.json", "r", encoding="utf-8") as f:
    cookies = json.load(f)

for cookie in cookies:
    cookie.pop("sameSite", None)   # remove unsupported key
    cookie.pop("sameSitePolicy", None)

    # If expirationDate exists, rename to expiry
    if "expirationDate" in cookie:
        try:
            cookie["expiry"] = int(cookie.pop("expirationDate"))
        except:
            cookie.pop("expirationDate", None)

    try:
        driver.add_cookie(cookie)
    except WebDriverException:
        # Retry without domain if it mismatches
        cookie.pop("domain", None)
        try:
            driver.add_cookie(cookie)
        except Exception:
            pass

# Refresh to apply cookies
driver.refresh()
time.sleep(3)

# Go directly to inbox
driver.get("https://nexizo.ai/app/tenders/inbox")
time.sleep(5)

print("Page title:", driver.title)
driver.save_screenshot("nexizo_inbox.png")


Page title: Nexizo


True

In [5]:
import os
import time
import json
import shutil
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ========== CONFIG ==========
DOWNLOAD_DIR = os.path.join(os.getcwd(), "temp_downloads")  # temp folder
LIVETENDER_DIR = r"C:\Users\Jaydeb\OneDrive - RASHMI METALIKS LIMITED\RML_DATA\Nexizo_Bidassist\Livetender"
PASTTENDER_DIR = r"C:\Users\Jaydeb\OneDrive - RASHMI METALIKS LIMITED\RML_DATA\Nexizo_Bidassist\Pasttender"
COOKIES_FILE = "cookies.json"
TODAY = datetime.today().strftime("%Y-%m-%d")

# ========== CHROME SETUP ==========
options = webdriver.ChromeOptions()
prefs = {"download.default_directory": DOWNLOAD_DIR,
         "download.prompt_for_download": False,
         "download.directory_upgrade": True,
         "safebrowsing.enabled": True}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(options=options)
driver.maximize_window()

# ========== LOAD COOKIES ==========
driver.get("https://nexizo.ai")
with open(COOKIES_FILE, "r", encoding="utf-8") as f:
    cookies = json.load(f)

for cookie in cookies:
    cookie.pop("sameSite", None)
    cookie.pop("sameSitePolicy", None)
    if "expirationDate" in cookie:
        try:
            cookie["expiry"] = int(cookie.pop("expirationDate"))
        except:
            cookie.pop("expirationDate", None)
    try:
        driver.add_cookie(cookie)
    except:
        cookie.pop("domain", None)
        try:
            driver.add_cookie(cookie)
        except:
            pass

driver.refresh()

wait = WebDriverWait(driver, 30)

# ========== HELPER FUNCTIONS ==========
def reset_all_if_present():
    try:
        reset_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//span[text()='Reset All']")))
        reset_btn.click()
        print("✅ Reset All clicked")
    except:
        print("⏭️ No Reset All found, skipping")

def click_checkbox_and_download():
    # Click checkbox
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input.fs-17[type='checkbox']"))).click()
    print("✅ Checkbox selected")

    # Click Download
    wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Download']"))).click()
    print("✅ Download clicked")

    # Wait for View Request
    view_request = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[text()='View Request']")))
    view_request.click()
    print("✅ View Request clicked")

    # Final Download button inside Request
    final_download = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[.//span[text()='Download']]")))
    final_download.click()
    print("✅ Final Download clicked")

    # Wait for file to appear in downloads
    time.sleep(10)

def move_latest_file(target_dir, prefix):
    files = os.listdir(DOWNLOAD_DIR)
    files = [os.path.join(DOWNLOAD_DIR, f) for f in files]
    latest_file = max(files, key=os.path.getctime)
    new_name = os.path.join(target_dir, f"{prefix}_{TODAY}.xlsx")
    shutil.move(latest_file, new_name)
    print(f"📂 File saved as: {new_name}")

# ========== STEP 1: INBOX ==========
driver.get("https://nexizo.ai/app/tenders/inbox")
reset_all_if_present()
click_checkbox_and_download()
move_latest_file(LIVETENDER_DIR, "LiveTenders")

# Go Back
wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Back']"))).click()
print("⬅️ Back clicked")

# ========== STEP 2: TENDER RESULTS ==========
wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Tender Results']"))).click()
print("✅ Tender Results opened")

reset_all_if_present()
click_checkbox_and_download()
move_latest_file(PASTTENDER_DIR, "PastTenders")

print("🎉 Process Completed Successfully")
driver.quit()


✅ Reset All clicked
✅ Checkbox selected
✅ Download clicked
✅ View Request clicked


TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff7828a78d5+2802741]
	GetHandleVerifier [0x0x7ff78260eb70+79568]
	(No symbol) [0x0x7ff7823ac0fa]
	(No symbol) [0x0x7ff782402a96]
	(No symbol) [0x0x7ff782402d4c]
	(No symbol) [0x0x7ff782456017]
	(No symbol) [0x0x7ff78242accf]
	(No symbol) [0x0x7ff782452e24]
	(No symbol) [0x0x7ff78242aa63]
	(No symbol) [0x0x7ff7823f3c91]
	(No symbol) [0x0x7ff7823f4a23]
	GetHandleVerifier [0x0x7ff7828d2ced+2979917]
	GetHandleVerifier [0x0x7ff7828cd0f3+2956371]
	GetHandleVerifier [0x0x7ff7828eacbd+3078173]
	GetHandleVerifier [0x0x7ff78262836e+184014]
	GetHandleVerifier [0x0x7ff78263024f+216495]
	GetHandleVerifier [0x0x7ff7826170c4+113700]
	GetHandleVerifier [0x0x7ff782617279+114137]
	GetHandleVerifier [0x0x7ff7825fdf78+10968]
	BaseThreadInitThunk [0x0x7ffaea1cdbe7+23]
	RtlUserThreadStart [0x0x7ffaeb485a4c+44]


In [2]:
import os
import time
import json
import shutil
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ================= CONFIG =================
DOWNLOAD_DIR = os.path.join(os.getcwd(), "temp_downloads")
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

LIVETENDER_DIR = r"C:\Users\Jaydeb\OneDrive - RASHMI METALIKS LIMITED\RML_DATA\Nexizo_Bidassist\Livetender"
os.makedirs(LIVETENDER_DIR, exist_ok=True)

PASTTENDER_DIR = r"C:\Users\Jaydeb\OneDrive - RASHMI METALIKS LIMITED\RML_DATA\Nexizo_Bidassist\PastTenders"
os.makedirs(PASTTENDER_DIR, exist_ok=True)

COOKIES_FILE = "cookies.json"
TODAY = datetime.today().strftime("%Y-%m-%d")

# ================= CHROME SETUP =================
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": DOWNLOAD_DIR,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)
driver.maximize_window()
wait = WebDriverWait(driver, 30)

# ================= LOAD COOKIES =================
driver.get("https://nexizo.ai")
with open(COOKIES_FILE, "r", encoding="utf-8") as f:
    cookies = json.load(f)

for cookie in cookies:
    cookie.pop("sameSite", None)
    cookie.pop("sameSitePolicy", None)
    if "expirationDate" in cookie:
        try:
            cookie["expiry"] = int(cookie.pop("expirationDate"))
        except:
            cookie.pop("expirationDate", None)
    try:
        driver.add_cookie(cookie)
    except:
        cookie.pop("domain", None)
        try:
            driver.add_cookie(cookie)
        except:
            pass

driver.refresh()
time.sleep(5)

# ================= FUNCTIONS =================
def reset_all_if_present():
    try:
        btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Reset All']")))
        btn.click()
        print("✅ Reset All clicked")
        time.sleep(2)
    except:
        print("⏭️ No Reset All found")

def move_file_to_target(file_path, target_dir, prefix):
    new_name = os.path.join(target_dir, f"{prefix}_{TODAY}.xlsx")
    shutil.move(file_path, new_name)
    print(f"📂 File moved to: {new_name}")

def download_after_one_refresh():
    """Wait a few seconds, refresh once, then click download"""
    time.sleep(5)  # wait before refresh
    driver.refresh()
    print("🔄 Page refreshed once")
    time.sleep(5)  # wait for page to load

    download_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Download')]")))
    download_btn.click()
    print("✅ Download clicked")

    # Wait for the downloaded file to appear
    start_time = time.time()
    while True:
        files = [os.path.join(DOWNLOAD_DIR, f) for f in os.listdir(DOWNLOAD_DIR) if f.endswith(".xlsx")]
        if files:
            latest_file = max(files, key=os.path.getctime)
            print(f"✅ File downloaded: {latest_file}")
            return latest_file
        time.sleep(2)
        if time.time() - start_time > 60:
            raise Exception("❌ Timeout waiting for downloaded file")

def click_checkbox_download_and_view():
    # Click checkbox
    checkbox = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input.fs-17[type='checkbox']")))
    checkbox.click()
    print("✅ Checkbox selected")
    time.sleep(1)

    # Click initial Download
    download_trigger = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Download']")))
    download_trigger.click()
    print("✅ Initial Download clicked")
    time.sleep(2)

    # Click View Request
    view_request = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[text()='View Request']")))
    view_request.click()
    print("✅ View Request clicked")
    time.sleep(2)

    # Refresh once and download
    file_path = download_after_one_refresh()
    return file_path

# ================= STEP 1: INBOX =================
driver.get("https://nexizo.ai/app/tenders/inbox")
reset_all_if_present()
live_file = click_checkbox_download_and_view()
move_file_to_target(live_file, LIVETENDER_DIR, "LiveTenders")

# Go back
wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Back']"))).click()
print("⬅️ Back clicked")
time.sleep(2)

# ================= STEP 2: TENDER RESULTS =================
wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Tender Results']"))).click()
reset_all_if_present()
past_file = click_checkbox_download_and_view()
move_file_to_target(past_file, PASTTENDER_DIR, "PastTenders")

print("🎉 Process Completed Successfully")
driver.quit()


✅ Reset All clicked
✅ Checkbox selected
✅ Initial Download clicked
✅ View Request clicked
🔄 Page refreshed once
✅ Download clicked
✅ File downloaded: C:\Users\Jaydeb\temp_downloads\TENDER_RESULT_63540440-1146-4480-a64e-b20ee2981823_2025-08-27.xlsx
📂 File moved to: C:\Users\Jaydeb\OneDrive - RASHMI METALIKS LIMITED\RML_DATA\Nexizo_Bidassist\Livetender\LiveTenders_2025-08-27.xlsx
⬅️ Back clicked
⏭️ No Reset All found
✅ Checkbox selected
✅ Initial Download clicked
✅ View Request clicked
🔄 Page refreshed once
✅ Download clicked
✅ File downloaded: C:\Users\Jaydeb\temp_downloads\TENDER_LISTING_a991f00e-f42c-49b7-a949-f701596667fc_2025-08-27.xlsx
📂 File moved to: C:\Users\Jaydeb\OneDrive - RASHMI METALIKS LIMITED\RML_DATA\Nexizo_Bidassist\PastTenders\PastTenders_2025-08-27.xlsx
🎉 Process Completed Successfully
